In [5]:
import pandas as pd
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
from src.routing.route_utils import compute_route_schedule, route_feasibility, route_duration, compute_insertion_cost
data_path = Path("../data/features/olist/delivery_jobs_dataset.csv")

In [8]:
jobs = pd.read_csv(
    data_path,
    parse_dates = [
        'ready_time',
        'due_date'
    ]
)

display(jobs.head())

,job_id,order_id,seller_id,customer_id,pickup_lat,pickup_lng,delivery_lat,delivery_lng,ready_time,due_date,service_time_min,demand
0,e481f51cbdc54678b7cc49136f2d6af7,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,9ef432eb6251297304e76186b10a928d,-23.680729,-46.444238,-23.576983,-46.587161,2017-10-02 11:07:15,2017-10-18,30,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,53cdb2fc8bc7dce0b6741e2150273451,289cdb325fb7e7f891c38608bf9e0962,b0830fb4747a6c6d20dea0b8c802d7ef,-19.807681,-43.980427,-12.177924,-44.660711,2018-07-26 03:24:27,2018-08-13,30,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,47770eb9100c2d0c44946d9cf07ec65d,4869f7a5dfa277a7dca6462dcf3b52b2,41ce2a54c0b03bf3443c3d931a367089,-21.363502,-48.229601,-16.745150,-48.514783,2018-08-08 08:55:23,2018-09-04,30,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,949d5b44dbf5de918fe9c16f97b45f8a,66922902710d126a0e7d26b0e3805106,f88197465ea7920adcdbec7375364d82,-19.837682,-43.924053,-5.774190,-35.271143,2017-11-18 19:45:59,2017-12-15,30,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,ad21c59c0840e6cb83a9ceb5573f8159,2c9e548be18521d1c43cde1c582c6de8,8ab97904e6daea8866dbdbc4fb7aad2c,-23.543395,-46.262086,-23.676370,-46.514627,2018-02-13 22:20:29,2018-02-26,30,1.0


In [13]:
# simulating a small courier route from samples
sample_jobs = jobs.sample(10, random_state = 2).copy()

# Sort by ready_time to simulate natural arrival
sample_jobs = (
    sample_jobs
    .sort_values(by = 'ready_time')
    .reset_index(drop=True)
)

# converting samples to route list
job_list = sample_jobs.to_dict('records')
job_list[0:2]

[{'job_id': '75de6a99721eba5ffbb4c4bd2fa0ee07',
  'order_id': '75de6a99721eba5ffbb4c4bd2fa0ee07',
  'seller_id': 'c3cfdc648177fdbbbb35635a37472c53',
  'customer_id': 'a1a724ba9559410e160d1b1e5b1b1ce5',
  'pickup_lat': -25.46995508695772,
  'pickup_lng': -49.289821060375864,
  'delivery_lat': -20.001771055992613,
  'delivery_lng': -44.0102000334043,
  'ready_time': Timestamp('2017-02-04 08:45:10'),
  'due_date': Timestamp('2017-03-09 00:00:00'),
  'service_time_min': 30,
  'demand': 1.0},
 {'job_id': '7ce450bb81ffdf0d90e7902f42de59ae',
  'order_id': '7ce450bb81ffdf0d90e7902f42de59ae',
  'seller_id': '6c177e38df6d3f34182b1f1d427231bf',
  'customer_id': 'a0c50d02890a8e5a08966030bdcf75db',
  'pickup_lat': -25.43014407428628,
  'pickup_lng': -49.28531986744621,
  'delivery_lat': -23.58291958812837,
  'delivery_lng': -46.56674927098573,
  'ready_time': Timestamp('2017-04-25 15:05:16'),
  'due_date': Timestamp('2017-05-19 00:00:00'),
  'service_time_min': 30,
  'demand': 1.0}]

In [15]:
# defining route start time -> start at first job's ready time
start_time = job_list[0]['ready_time']
start_time

Timestamp('2017-02-04 08:45:10')

In [25]:
# building a route for testing from samples
route = job_list[0:7]

# testing route schedule
feasible, schedule = compute_route_schedule(route, start_time)
print(f"Route Feasibility: {feasible}")

if feasible:
    display(pd.DataFrame.from_dict(schedule))

# route duration
duration = route_duration(route, start_time)
print(f"Route duration (in minutes): {duration}")

# route feasibility
route_feasible = route_feasibility(route, start_time)
print(f"Feasibility Check for a route: {route_feasible}")

Route Feasibility: True


,job_id,arrival_time,departure_time
0,75de6a99721eba5ffbb4c4bd2fa0ee07,2017-02-04 08:45:10,2017-02-04 09:15:10
1,7ce450bb81ffdf0d90e7902f42de59ae,2017-04-25 15:05:16,2017-04-25 15:35:16
2,fecb65750b4fe05b8257f650b2e114a2,2017-11-03 13:07:18,2017-11-03 13:37:18
3,8574762135a91d4e44956dadbaacdd0c,2017-11-25 23:50:29,2017-11-26 00:20:29
4,9ae081992e9b8d305a8959fa5507ffe9,2017-12-29 02:09:10,2017-12-29 02:39:10
5,add65839c01cc30f0b205ae60ac6d508,2018-02-11 00:15:28,2018-02-11 00:45:28
6,30dbb7bb7d4c27c3efd6ab7c9c27f9d0,2018-04-04 03:29:28,2018-04-04 03:59:28


Route duration (in minutes): 610274.3
Feasibility Check for a route: True


In [ ]:
# checking the insertion cost for different positions
new_job = job_list[len(route)]
best_cost = float('inf')
best_post = None

for pos in range(len(route) + 1):
    cost = compute_insertion_cost(
        route,
        new_job,
        pos,
        start_time
    )

    if cost < best_cost:
        best_cost = cost
        best_pos = pos

    print(f"Inserting new job at position {pos} -> cost: {cost:.2f}")
    
print(f"\nBest position is {best_post} with a cost of {best_cost:.2f} to insert the new job in sample route")

Inserting new job at position 0 -> cost: inf
Inserting new job at position 1 -> cost: inf
Inserting new job at position 2 -> cost: inf
Inserting new job at position 3 -> cost: inf
Inserting new job at position 4 -> cost: inf
Inserting new job at position 5 -> cost: inf
Inserting new job at position 6 -> cost: 3671.09
Inserting new job at position 7 -> cost: 2304.99

Best position is None with a cost of 2304.99 to insert the new job in sample route
